##Understaning the Rösslers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sg
from scipy.signal import hilbert
from scipy.stats import shapiro, probplot
import seaborn as sns


def rossler(t, state, wx=1, wy=1, eyx=0, exy=0):
    """
    Computes the derivative of the Rössler dynamics given a state.
    """
    x1, x2, x3, y1, y2, y3 = state

    dx1dt = -wx*x2 - x3 + eyx*(y1-x1)
    dx2dt = wx*x1 + 0.165*x2
    dx3dt = 0.2 + x3 * (x1 - 10)

    dy1dt = -wy*y2 - y3 + exy*(x1-y1)
    dy2dt = wy*y1 + 0.165*y2
    dy3dt = 0.2 + y3 * (y1 - 10)

    return [dx1dt, dx2dt, dx3dt, dy1dt, dy2dt, dy3dt]

In [ ]:
def fourier_phase_randomized(x:np.ndarray, N:int=1, seed:int=None)->np.ndarray:
    """
    Generate N surrogates of an input signal on the time domain.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray[n_samples, N]
        Surrogates of the input signal.
    """

    rng = np.random.default_rng(seed)

    n = len(x)
    half = n // 2
    X_amp = np.abs(np.fft.fft(x))

    Yf = np.zeros((n, N), dtype=complex)
    Yf[0] = X_amp[0] # DC component

    phases = rng.uniform(0, 2*np.pi, size=(half - 1, N))
    Yf[1:half] = X_amp[1:half, None] * np.exp(1j * phases)

    if n % 2 == 0:
        Yf[half] = X_amp[half] # Nyquist frequency
        Yf[half+1:] = np.conj(Yf[1:half][::-1])
    else:
        Yf[half+1:] = np.conj(Yf[1:half+1][::-1])

    y = np.real(np.fft.ifft(Yf, axis=0))

    return y


def IAAFT(x:np.ndarray, N:int=1, n_iter:int=10, seed:int=None)->np.ndarray:
    """
    Generate N IAAFT surrogates for signal x.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates.
    n_iter : int
        Number of IAAFT iterations.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray [n_samples, N]
        IAAFT surrogate signals.
    """

    x_sorted = np.sort(x)
    X_amp = np.abs(np.fft.fft(x))

    y = fourier_phase_randomized(x, N, seed)

    for _ in range(n_iter):
        # Impose the exact time-domain amplitude distribution
        idx = np.argsort(y, axis=0)
        for k in range(N):
            y[idx[:, k], k] = x_sorted # sort y and replace with sorted original x

        # Impose Fourier amplitudes but keep phases
        Yf = np.fft.fft(y, axis=0)
        phases = Yf / (np.abs(Yf) + 1e-15)
        Yf = X_amp[:, None] * phases # impose original amplitude spectrum
        y = np.real(np.fft.ifft(Yf, axis=0))

    return y


def randomize_phase_velocity(phase:np.ndarray, N:int=1, seed:int=None, return_idx:bool=False)->np.ndarray:
    """
    Creates surrogates of a phase array by permuting the phase velocities.

    Parameters
    ----------
    phase : np.ndarray [n_samples]
        Hilbert phase.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    randomized_phase : np.ndarray[n_samples, N]
        Surrogates of the phase.
    """

    rng = np.random.default_rng(seed)
    n = len(phase)

    velocity = np.diff(phase)  # (n-1,)

    idx = np.column_stack([
        rng.permutation(n - 1) for _ in range(N)
    ])

    velocity_perm = velocity[idx]

    randomized_phase = np.zeros((n, N))
    randomized_phase[0, :] = phase[0]
    randomized_phase[1:, :] = phase[0] + np.cumsum(velocity_perm, axis=0)

    if return_idx:
        return randomized_phase, idx
    else:
        return randomized_phase


def hilbert_velocity_randomized(x:np.ndarray, N:int=1, seed:int=None, return_hilbert:bool=False)->np.ndarray:
    """
    Generate N surrogates of an input signal on the Hilbert phase domain.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray[n_samples, N]
        Surrogates of the input signal.
    """

    analytical = hilbert(x)
    amplitude = np.abs(analytical)
    phase = np.unwrap(np.angle(analytical))

    randomized_phase = randomize_phase_velocity(phase, N, seed)

    y = amplitude[:, None] * np.cos(randomized_phase)

    if return_hilbert:
        return y, phase, randomized_phase, amplitude
    else:
        return y

def comute_omega(fase_unwrapped, fs):
    dt = 1.0 / fs

    omega = np.diff(fase_unwrapped) / dt
    return omega


def compute_v(fase_unwrapped, fs):
    """
    Calcula el coeficiente de variación de la velocidad de fase (V).

    Args:
        fase_unwrapped (array): La fase desenvuelta de la señal.
        fs (int): Frecuencia de muestreo en Hz.

    Returns:
        dict: Contiene los valores de V, M y S.
    """
    dt = 1.0 / fs

    omega = np.diff(fase_unwrapped) / dt

    M = np.mean(omega)

    S = np.std(omega)

    V = S / M

    return {
        "V_irregularidad": V,
        "M_velocidad_media": M,
        "S_desviacion_estandar": S,
        "omega_instantanea": omega
    }

def calcular_metricas_fase(x, fs):
    """
    Calcula V, M y S siguiendo las ecuaciones (7), (8) y (9) del paper.
    [cite: 197, 199, 200, 206]
    """
    dt = 1.0 / fs
    fase_unwrapped = np.unwrap(np.angle(hilbert(x)))

    omega = np.diff(fase_unwrapped) / dt

    M = np.mean(omega)
    S = np.std(omega)
    V = S / M

    return V, M, S

def test_hipotesis_espinozo(x, fs, n_surrogates=19):
    """
    Realiza el test de hipótesis univariado siguiendo a Espinoso (2022).
    Devuelve los valores individuales de los surrogates para graficar.
    """
    # 1. Calcular métricas de la señal original
    v_orig, m_orig, s_orig = calcular_metricas_fase(x, fs)

    # 2. Generar N surrogates (H0: Proceso lineal estacionario)
    surrogates = IAAFT(x, N=n_surrogates)

    v_surr, m_surr, s_surr = [], [], []

    # Calculamos métricas para cada surrogate
    for i in range(n_surrogates):
        v, m, s = calcular_metricas_fase(surrogates[:, i], fs)
        v_surr.append(v)
        m_surr.append(m)
        s_surr.append(s)

    # 3. Criterio de rechazo de dos colas:
    # Se rechaza si el original es MENOR que el mínimo O MAYOR que el máximo

    # Lógica para M (Velocidad de fase media)
    rechazo_M_bajo = m_orig < np.min(m_surr) # Caso típico en focales según Espinoso
    rechazo_M_alto = m_orig > np.max(m_surr) # Caso de velocidad inusualmente alta
    rechazo_H0_M = rechazo_M_bajo or rechazo_M_alto

    # Lógica para V (Coeficiente de variación)
    rechazo_V_bajo = v_orig < np.min(v_surr)
    rechazo_V_alto = v_orig > np.max(v_surr)
    rechazo_H0_V = rechazo_V_bajo or rechazo_V_alto

    return {
        "original": {"V": v_orig, "M": m_orig, "S": s_orig},
        "surrogates_stats": {
            "V_mean": np.mean(v_surr),
            "M_mean": np.mean(m_surr),
            "V_min": np.min(v_surr),
            "V_max": np.max(v_surr),
            "M_min": np.min(m_surr),
            "M_max": np.max(m_surr)
        },
        "lista_surr_V": v_surr,
        "lista_surr_M": m_surr,
        "rechazo_H0_M": rechazo_H0_M,
        "rechazo_H0_V": rechazo_H0_V,
        "detalle_rechazo": {
            "M_tipo": "bajo" if rechazo_M_bajo else ("alto" if rechazo_M_alto else "ninguno"),
            "V_tipo": "bajo" if rechazo_V_bajo else ("alto" if rechazo_V_alto else "ninguno")
        }
    }


def graficar_test_surrogates(res_test, metrica='V', nombre_senal="Señal AR"):
    """
    Crea un histograma de los valores de los surrogates y marca el valor original.

    Args:
        res_test (dict): El diccionario devuelto por la función test_hipotesis_espinozo.
        metrica (str): 'V' o 'M' para elegir qué métrica visualizar.
        nombre_senal (str): Etiqueta para la gráfica.
    """
    # Extraer datos
    val_orig = res_test['original'][metrica]
    # Necesitaremos que la función de test devuelva la lista completa de valores
    # de los surrogates para graficar la distribución.
    # (Asegúrate de guardar 'v_surr' o 'm_surr' en el dict de salida del test).
    val_surr = res_test['lista_surr_' + metrica]

    plt.figure(figsize=(8, 5))

    # Graficar la distribución de los surrogates (procesos lineales)
    sns.histplot(val_surr, kde=True, color='grey', label='Distribución Surrogates (H0)', alpha=0.6)

    # Marcar el valor de la señal original
    plt.axvline(val_orig, color='red', linestyle='--', linewidth=2, label=f'Original ({metrica}={val_orig:.4f})')

    # Títulos y etiquetas siguiendo el estilo del paper
    plt.title(f'Test de Hipótesis: {nombre_senal} ({metrica})')
    plt.xlabel(f'Valor de {metrica}')
    plt.ylabel('Frecuencia / Recuento')

    # Indicar si hubo rechazo
    rechazo = res_test[f'rechazo_H0_{metrica}']
    resultado_texto = "RECHAZADO (P < 0.05)" if rechazo else "NO RECHAZADO"
    plt.annotate(f'Resultado: {resultado_texto}',
                 xy=(0.05, 0.9), xycoords='axes fraction',
                 fontsize=12, fontweight='bold', color='red' if rechazo else 'black')

    plt.legend()
    plt.show()

def compare_signals(fases_unwrapped_1, fases_unwrapped_2, fs, nombre1="AR 1", nombre2="AR 2"):
    """
    Calcula métricas y compara estadísticamente dos grupos de señales.
    """
    dt = 1.0 / fs

    # Función interna para métricas
    def obtener_metricas(fase):
        omega = np.diff(fase) / dt
        m = np.mean(omega)
        s = np.std(omega)
        v = s / m
        return v, m, s

    # Si tienes múltiples épocas o realizaciones, calculamos para cada una
    # (Asumiendo que fases_unwrapped es una lista de arrays)
    v1 = [obtener_metricas(f)[0] for f in fases_unwrapped_1]
    v2 = [obtener_metricas(f)[0] for f in fases_unwrapped_2]

    # Test estadístico (Mann-Whitney U)
    stat, p_value = stats.mannwhitneyu(v1, v2)

    print(f"--- Resultados de Comparación ($V$) ---")
    print(f"Media {nombre1}: {np.mean(v1):.4f}")
    print(f"Media {nombre2}: {np.mean(v2):.4f}")
    print(f"Valor p: {p_value:.5f}")

    if p_value < 0.05:
        print("¡Resultado Significativo! Las señales presentan niveles de irregularidad distintos.")
    else:
        print("No se encontraron diferencias significativas.")

    return v1, v2



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sg
from scipy.signal import hilbert
from scipy.integrate import solve_ivp


# Parameters
fs = 5       # sampling frequency
T = 300         # total time in seconds
n_samples = T * fs
t = np.linspace(0, T, n_samples)

cutoff = 9.0
order = 4
#b, a = sg.butter(order, cutoff, fs=fs, btype='low')

# Initial condition
state0 = [1, 0, 0, 1, 0, 0]

# Simulate Rössler system
sol = solve_ivp(lambda t, y: rossler(t, y), [0, T], state0, t_eval=t, method='RK45')
x1 = sol.y[0]  # only the first component
#x1 = x1 + np.random.normal(0, 0.6, size=len(x1))

# Optional: low-pass filter
# x_filt = sg.filtfilt(b, a, x1)

# PSD
freqs, Pxx = sg.periodogram(
    x1,
    fs=fs,
    window='hann',
    scaling='density'
)

plt.figure(figsize=(12,4))
plt.plot(freqs, 10*np.log10(Pxx + 1e-12))
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD [dB/Hz]")
plt.title("PSD — Rössler x1 component")
plt.grid(True)
plt.tight_layout()
plt.show()

# Time series
plt.figure(figsize=(12,4))
plt.plot(t, x1, label='x1')
plt.title("Rössler dynamics — x1 component")
plt.xlabel("Time [s]")
plt.ylabel("x1")
plt.legend()
plt.tight_layout()
plt.show()

# Phase
phase = np.angle(hilbert(x1))
plt.figure(figsize=(12,4))
plt.hist(phase, bins=100, density=True)
plt.xlabel("Phase (rad)")
plt.ylabel("Probability density")
plt.title("Phase Distribution — x1 component")
plt.grid(True)
plt.tight_layout()
plt.show()

# Instantaneous frequency (optional)
phase_unwrapped = np.unwrap(phase)

dphi = np.diff(phase)

plt.figure(figsize=(12,4))
plt.hist(dphi, bins=100)
plt.xlabel("Rad")
plt.ylabel("Count")
plt.title(f"Phase increment distribution")
plt.grid(True)
plt.show()

omega = np.gradient(phase_unwrapped, 1/fs)
plt.figure(figsize=(12,4))
plt.plot(t, phase_unwrapped, label='Instantaneous frequency', color='red')
plt.xlabel("Time [s]")
plt.ylabel("Angular frequency [rad/s]")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import hilbert
from scipy.spatial.distance import pdist, squareform
import pywt

# -----------------------------
# Rössler system
# -----------------------------
def rossler(t, state, wx=1, wy=1, eyx=0, exy=0):
    x1, x2, x3, y1, y2, y3 = state
    dx1dt = -wx*x2 - x3 + eyx*(y1-x1)
    dx2dt = wx*x1 + 0.165*x2
    dx3dt = 0.2 + x3*(x1 - 10)
    dy1dt = -wy*y2 - y3 + exy*(x1-y1)
    dy2dt = wy*y1 + 0.165*y2
    dy3dt = 0.2 + y3*(y1 - 10)
    return [dx1dt, dx2dt, dx3dt, dy1dt, dy2dt, dy3dt]

# -----------------------------
# Simulation parameters
# -----------------------------
fs = 0.6    # sampling frequency
T = 300        # total time [s]
n_samples = 1000
t = np.linspace(0, T, n_samples)

state0 = [1, 0, 0, 1, 0, 0]

sol = solve_ivp(lambda t, y: rossler(t, y), [0, T], state0, t_eval=t, method='RK45')
x1, x2, x3 = sol.y[0], sol.y[1], sol.y[2]
print(sol.t)

#x1 = x1 + np.random.normal(0, 0.6, size=len(x1))

# -----------------------------
# 1️⃣ Time series x1
# -----------------------------
plt.figure(figsize=(12,4))
plt.plot(t, x1, color='blue')
plt.title("Rössler dynamics — x1 component")
plt.xlabel("Time [s]")
plt.ylabel("x1")
plt.grid(True)
plt.tight_layout()
plt.show()

# -----------------------------
# 2️⃣ Phase-space plot x1 vs x2
# -----------------------------
plt.figure(figsize=(6,6))
plt.plot(x1, x2, color='purple')
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Phase-space trajectory")
plt.grid(True)
plt.tight_layout()
plt.show()


# -----------------------------
# 5️⃣ Wavelet scalogram (time-frequency)
# -----------------------------

# -----------------------------
# Optional: Hilbert amplitude & phase
# -----------------------------
analytic_signal = hilbert(x1)


x_real = np.real(analytic_signal)
x_imag = np.imag(analytic_signal)

N = 50
#x_real = x_real[5:N]
#x_imag = x_imag[5:N]

plt.figure(figsize=(4,4))
plt.plot(x_real, x_imag, lw=0.6)
plt.scatter(x_real[0], x_imag[0], color='green', label='Start', s=40)
plt.scatter(x_real[-1], x_imag[-1], color='red', label='End', s=40)
plt.xlabel("Real part")
plt.ylabel("Hilbert transform")
plt.title(f"Analytic signal trajectory — φ")
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

amplitude_envelope = np.abs(analytic_signal)
phase = np.angle(analytic_signal)
unwraped = np.unwrap(phase)

plt.figure(figsize=(12,4))
plt.plot(t, amplitude_envelope, label='Amplitude envelope', color='orange')
plt.title("Hilbert amplitude of x1")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(t, unwraped, color='red')
#plt.hist(phase, bins=100, density=True, color='red')
plt.xlabel("Phase (rad)")
plt.ylabel("Probability density")
plt.title("Hilbert phase distribution of x1")
plt.grid(True)
plt.tight_layout()
plt.show()

dphi = np.diff(phase_unwrapped)

plt.figure(figsize=(12,4))
plt.hist(dphi, bins=100)
plt.ylabel("Count")
plt.title(f"Phase increment distribution")
plt.grid(True)
plt.show()

_,s,_ = calcular_metricas_fase(x1, fs)
print(f"Velocidad media: {s}")

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

def generate_rossler_xyz(wx=1.0, fs=200, T=100, a=0.15, b=0.2, c=10):
    """
    Generate Rössler x(t), y(t), z(t) signals.

    Parameters:
        wx : float - scaling factor for initial conditions
        fs : int   - sampling frequency (Hz)
        T  : float - total time (seconds)
        a,b,c : float - Rössler parameters
    Returns:
        t : array - time vector
        x : array - x(t)
        y : array - y(t)
        z : array - z(t)
    """
    t = np.linspace(0, T, int(T*fs))

    def rossler(t, s):
        x, y, z = s
        return [-y - z, x + a*y, b + z*(x - c)]

    # initial conditions scaled by wx
    s0 = [wx, wx, wx]
    sol = solve_ivp(rossler, (0, T), s0, t_eval=t)

    x, y, z = sol.y
    return t, x, y, z


def generate_rossler_x1(wx=1.0, fs=200, T=100):
    """Return only x(t)"""
    t, x, _, _ = generate_rossler_xyz(wx, fs, T)
    return x

def generate_rossler_z1(wx=1.0, fs=200, T=100):
    """Return only z(t)"""
    t, _, _, z = generate_rossler_xyz(wx, fs, T)
    return z


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import hilbert

def rossler(t, s, a=0.15, b=0.2, c=10):
    x, y, z = s
    return [-y - z, x + a*y, b + z*(x - c)]

def phase_velocity(x, fs):
    phi = np.unwrap(np.angle(hilbert(x)))
    omega = np.diff(phi) * fs
    return omega

# --- simulate system ---
t = np.linspace(0, 200, 20000)
fs = len(t) / (t[-1] - t[0])
sol = solve_ivp(rossler, (0, 200), [1, 1, 1], t_eval=t)

x, z = sol.y[0], sol.y[2]

# add noise
x = x + np.random.normal(0, 0.1, size=len(x))

# compute phase velocity
omega = phase_velocity(x, fs)
t_omega = t[:-1]

# --- compute mean and std in 10-second windows ---
window_size = 10  # seconds
window_samples = int(window_size * fs)
local_means = []
local_stds = []
window_times = []

for start in range(0, len(omega), window_samples):
    end = min(start + window_samples, len(omega))
    local_window = omega[start:end]
    local_means.append(local_window.mean())
    local_stds.append(local_window.std())
    window_times.append(t_omega[start:end].mean())

# --- global metrics ---
M_global = omega.mean()
S_global = omega.std()

# --- plotting ---
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(t, x)
plt.ylabel("x(t)")

plt.subplot(3, 1, 2)
plt.plot(t, z)
plt.ylabel("z(t)")

plt.subplot(3, 1, 3)
plt.plot(t_omega, omega, alpha=0.5, label="ω(t)")
plt.axhline(M_global, color='red', linestyle='--', label="Global mean")
plt.step(window_times, local_means, where='mid', color='orange', label="Local 10s mean")
plt.fill_between(window_times,
                 np.array(local_means) - np.array(local_stds),
                 np.array(local_means) + np.array(local_stds),
                 color='orange', alpha=0.3, label="Local ±1 std")
plt.ylabel("ω(t)")
plt.xlabel("time")
plt.legend()
plt.title("Mean and variability of ω in 10-second windows")

plt.tight_layout()
plt.show()

# --- print mean and std for each window ---
print("Window start time (s) | Mean ω | Std ω")
for i, (mean_val, std_val, t_center) in enumerate(zip(local_means, local_stds, window_times)):
    print(f"{i*10:3d}-{i*10+10:3d} s | {mean_val:7.3f} | {std_val:7.3f}")


#Noise + filtering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# ----- Baseline (no noise) -----
x1 = x1.copy()
x1 = x1[100:900]
a_sig_clean = hilbert(x1)
p_clean = np.unwrap(np.angle(a_sig_clean))
o_clean = np.diff(p_clean)
V_clean = np.std(o_clean) / np.mean(o_clean)

# ----- Noise experiment -----
noise_levels = np.linspace(0, 8, 20)
results_v = []

for level in noise_levels:
    noisy_signal = x1 + np.random.normal(0, level, size=len(x1))

    a_sig = hilbert(noisy_signal)
    p = np.unwrap(np.angle(a_sig))
    o = np.diff(p)

    V = np.std(o) / np.mean(o)
    results_v.append(V)

# ----- Plot comparison -----
plt.figure(figsize=(8,5))

plt.plot(noise_levels, results_v, 'o-', label="With noise")
plt.axhline(V_clean, color='black', linestyle='--',
            label="No noise (baseline)")

plt.xlabel("Noise level (std)")
plt.ylabel("Phase Irregularity (V)")
plt.title("Impact of Noise on Phase Dynamics")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('ruido_vs_v.png')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import hilbert, butter, filtfilt

# =====================================================
# 1. RÖSSLER SYSTEM
# =====================================================

def rossler(t, state, wx=1, wy=1, eyx=0, exy=0):
    x1, x2, x3, y1, y2, y3 = state
    dx1dt = -wx*x2 - x3 + eyx*(y1-x1)
    dx2dt = wx*x1 + 0.165*x2
    dx3dt = 0.2 + x3*(x1 - 10)
    dy1dt = -wy*y2 - y3 + exy*(x1-y1)
    dy2dt = wy*y1 + 0.165*y2
    dy3dt = 0.2 + y3*(y1 - 10)
    return [dx1dt, dx2dt, dx3dt, dy1dt, dy2dt, dy3dt]

# =====================================================
# 2. SIMULATION
# =====================================================

T = 200
n_samples = 5000
t = np.linspace(0, T, n_samples)

fs = n_samples / T  # TRUE sampling frequency

state0 = [1, 0, 0, 1, 0, 0]

sol = solve_ivp(lambda t, y: rossler(t, y),
                [0, T], state0,
                t_eval=t)

raw_signal = sol.y[0]

# Remove transient
raw_signal = raw_signal[500:]
t = t[500:]

# =====================================================
# 3. BASELINE PHASE METRIC
# =====================================================

def phase_irregularity(x, fs):
    dt = 1.0 / fs
    analytic = hilbert(x)
    phase = np.unwrap(np.angle(analytic))
    omega = np.diff(phase) / dt
    return np.std(omega) / np.mean(omega)

V_clean = phase_irregularity(raw_signal, fs)

# =====================================================
# 4. FILTER SETTINGS
# =====================================================

cutoffs = [0.5, 1, 2, 4]   # Hz (adapted to Rössler scale)
order = 4

noise_levels = np.linspace(0, 2, 20)

# =====================================================
# 5. NOISE + FILTER STUDY
# =====================================================

plt.figure(figsize=(9,6))

for cutoff in cutoffs:

    b, a = butter(order, cutoff / (fs/2), btype='low')

    results_v = []

    for level in noise_levels:

        noise = np.random.normal(0, level, size=len(raw_signal))
        noisy_signal = raw_signal + noise

        filtered = filtfilt(b, a, noisy_signal)

        V = phase_irregularity(filtered, fs)
        results_v.append(V)

    plt.plot(noise_levels, results_v, 'o-', label=f'LPF cutoff = {cutoff} Hz')

# Baseline reference
plt.axhline(V_clean, color='black', linestyle='--',
            label="No noise baseline")

plt.xlabel("Noise level (std)")
plt.ylabel("Phase Irregularity (V)")
plt.title("Rössler: Noise + Low-pass Filtering Impact")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
from scipy.integrate import solve_ivp

# ---------------------------------------------------
# Helper: smooth transitions (sigmoid)
# ---------------------------------------------------
def smooth_step(t, t0, width=2):
    return 1 / (1 + np.exp(-(t - t0) / width))


# ---------------------------------------------------
# 1. Rössler with NORMAL → SEIZURE → NORMAL
# ---------------------------------------------------
def sistema_seizure_sim(t_total, fs):
    t = np.linspace(0, t_total, int(t_total * fs))

    t1 = t_total / 3
    t2 = 2 * t_total / 3

    def smooth_step(t, t0, width=1.5):
        return 1 / (1 + np.exp(-(t - t0) / width))

    def rossler_dinamico(t_curr, s):
        x, y, z = s

        # seizure activation (0 → 1 → 0)
        up = smooth_step(t_curr, t1)
        down = smooth_step(t_curr, t2)
        seizure = up * (1 - down)

        # -------------------------------
        # PARAMETER CHANGES DURING SEIZURE
        # -------------------------------

        # more unstable dynamics
        a_normal = 0.18
        a_seizure = 0.35
        a_curr = a_normal * (1 - seizure) + a_seizure * seizure

        # faster oscillations
        w_normal = 1.0
        w_seizure = 2.5
        w = w_normal * (1 - seizure) + w_seizure * seizure

        # stronger nonlinear drive → higher amplitude
        drive = 10 - 4 * seizure

        # burst-like excitation (EEG spikes)
        burst_noise = seizure * 2.5 * np.random.randn()

        return [
            -y - z + burst_noise,
            w * x + a_curr * y,
            0.2 + z * (x - drive)
        ]

    sol = solve_ivp(rossler_dinamico, (0, t_total), [1, 1, 1], t_eval=t)

    # amplitude explosion during seizure (EEG-like)
    envelope = 1 + 3 * (
        smooth_step(t, t1) * (1 - smooth_step(t, t2))
    )

    return t, sol.y[0] * envelope


# ---------------------------------------------------
# 2. Bandpass filter → better phase estimation
# ---------------------------------------------------
def bandpass(signal, fs, low=0.5, high=8):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype="band")
    return filtfilt(b, a, signal)


# ---------------------------------------------------
# 3. Moving window analysis
# ---------------------------------------------------
def analizar_evolucion(t, x, window_size, fs):
    step = int(fs)
    win_len = int(window_size * fs)

    tiempos_centrales = []
    m_values, s_values, v_values = [], [], []

    for i in range(0, len(x) - win_len, step):
        segmento = x[i:i + win_len]

        # filter before Hilbert → HUGE improvement
        #segmento = bandpass(segmento, fs)

        dt = 1 / fs

        analytic = hilbert(segmento)
        phi = np.unwrap(np.angle(analytic))

        omega = np.gradient(phi) / dt

        M = np.mean(omega)
        S = np.std(omega)

        # avoid division instability
        V = S / np.abs(M) if np.abs(M) > 1e-6 else 0

        tiempos_centrales.append(t[i + win_len // 2])
        m_values.append(M)
        s_values.append(S)
        v_values.append(V)

    return (
        np.array(tiempos_centrales),
        np.array(m_values),
        np.array(s_values),
        np.array(v_values)
    )


# ---------------------------------------------------
# Helper: smooth curves for visualization
# ---------------------------------------------------
def moving_average(x, w=5):
    return np.convolve(x, np.ones(w)/w, mode="same")


# ---------------------------------------------------
# --- RUN ---
# ---------------------------------------------------
fs = 1
t_total = 300

t, x = sistema_seizure_sim(t_total, fs)

# moderate noise (less destructive)
x_noisy = x + np.random.normal(0, 2, size=len(x))

t_ev, M_ev, S_ev, V_ev = analizar_evolucion(t, x_noisy, 10, fs)

# smooth metrics for clearer trends
M_ev = moving_average(M_ev)
S_ev = moving_average(S_ev)
V_ev = moving_average(V_ev)


# ---------------------------------------------------
# --- VISUALIZATION ---
# ---------------------------------------------------
plt.figure(figsize=(12, 9))

t1 = t_total / 3
t2 = 2 * t_total / 3

# Signal
plt.subplot(3, 1, 1)
plt.plot(t, x_noisy, color="gray", alpha=0.5)
plt.axvline(t1, color="green", linestyle="--", label="Seizure start")
plt.axvline(t2, color="blue", linestyle="--", label="Return to normal")
plt.title("Rössler Signal: Normal → Seizure → Normal")
plt.ylabel("Amplitude")
plt.legend()

# M and S
plt.subplot(3, 1, 2)
plt.plot(t_ev, M_ev/np.mean(M_ev), label="M (Phase velocity)", linewidth=2)
plt.plot(t_ev, S_ev/np.mean(S_ev), label="S (Dispersion)", linewidth=2)
plt.title("Phase Metrics Evolution")
plt.ylabel("Normalized")
plt.legend()

# V
plt.subplot(3, 1, 3)
plt.plot(t_ev, V_ev, linewidth=2, label="V = S/M")
plt.title("Irregularity Collapse During Seizure")
plt.xlabel("Time (s)")
plt.ylabel("V")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d

# =====================================================
# 1. RÖSSLER SYSTEM
# =====================================================

def rossler(t, state, a=0.165, b=0.2, c=5.7):
    x, y, z = state
    return [-y - z, x + a*y, b + z*(x - c)]

t = np.linspace(0, 300, 30000)
dt = t[1] - t[0]
fs = 1 / dt   # ✅ define sampling frequency

sol = solve_ivp(
    rossler,
    [t[0], t[-1]],
    [1.0, 1.0, 1.0],
    t_eval=t
)

raw_signal = sol.y[0]   # ✅ define raw_signal

# =====================================================
# 2. REMOVE TRANSIENT (CONSISTENTLY)
# =====================================================

cut = int(5 * fs)
raw_signal = raw_signal[cut:]
t = t[cut:]   # ✅ also cut time vector

# =====================================================
# 3. ADD NOISE
# =====================================================

noise_level = 0.007
noise = noise_level * np.random.randn(len(raw_signal))
noisy_signal = raw_signal + noise

# =====================================================
# 4. FILTERED VERSION
# =====================================================

filtered_signal = gaussian_filter1d(noisy_signal, sigma=2)

# =====================================================
# 5. PHASE METRICS
# =====================================================

def calcular_metricas_fase(x, fs):

    dt = 1.0 / fs
    fase_unwrapped = np.unwrap(np.angle(hilbert(x)))
    omega = np.diff(fase_unwrapped) / dt

    M = np.mean(omega)
    S = np.std(omega)
    V = S / M

    return V, M, S, omega

# =====================================================
# 6. COMPUTE METRICS
# =====================================================

V_raw, M_raw, S_raw, omega_raw = calcular_metricas_fase(raw_signal, fs)
V_noisy, M_noisy, S_noisy, omega_noisy = calcular_metricas_fase(noisy_signal, fs)
V_filt, M_filt, S_filt, omega_filt = calcular_metricas_fase(filtered_signal, fs)

# =====================================================
# 7. PRINT RESULTS
# =====================================================

print("\n--- PHASE VELOCITY METRICS ---\n")

print("RAW:")
print(f"V = {V_raw:.4f},  M = {M_raw:.4f},  S = {S_raw:.4f}\n")

print("NOISY:")
print(f"V = {V_noisy:.4f},  M = {M_noisy:.4f},  S = {S_noisy:.4f}\n")

print("FILTERED:")
print(f"V = {V_filt:.4f},  M = {M_filt:.4f},  S = {S_filt:.4f}\n")

# =====================================================
# 8. PLOTS
# =====================================================

plt.figure(figsize=(14,10))

# Signals
plt.subplot(3,1,1)
plt.plot(t, raw_signal, label="Raw")
plt.plot(t, noisy_signal, alpha=0.6, label="Noisy")
plt.plot(t, filtered_signal, linewidth=2, label="Filtered")
plt.title("Signals")
plt.legend()

# Instantaneous frequency
plt.subplot(3,1,2)
plt.plot(t[1:], omega_raw, label="Omega Raw")
plt.plot(t[1:], omega_noisy, alpha=0.6, label="Omega Noisy")
plt.plot(t[1:], omega_filt, linewidth=2, label="Omega Filtered")
plt.title("Instantaneous Phase Velocity ω(t)")
plt.legend()

# Bar comparison
plt.subplot(3,1,3)
labels = ["Raw", "Noisy", "Filtered"]
V_vals = [V_raw, V_noisy, V_filt]
M_vals = [M_raw, M_noisy, M_filt]

x = np.arange(len(labels))

plt.bar(x - 0.2, V_vals, width=0.4, label="V")
plt.bar(x + 0.2, M_vals, width=0.4, label="M")
plt.xticks(x, labels)
plt.title("Metric Comparison")
plt.legend()

plt.tight_layout()
plt.show()

# =====================================================
# 9. ZOOM ON INSTANTANEOUS FREQUENCY
# =====================================================

t_start = 80
t_end   = 180

idx_start = np.argmin(np.abs(t - t_start))
idx_end   = np.argmin(np.abs(t - t_end))

plt.figure(figsize=(12,6))

plt.plot(t[1:][idx_start:idx_end],
         omega_raw[idx_start:idx_end],
         label="Omega Raw")

plt.plot(t[1:][idx_start:idx_end],
         omega_noisy[idx_start:idx_end],
         alpha=0.6,
         label="Omega Noisy")

plt.plot(t[1:][idx_start:idx_end],
         omega_filt[idx_start:idx_end],
         linewidth=2,
         label="Omega Filtered")

plt.title(f"Zoomed Instantaneous Phase Velocity ω(t)\n[{t_start}s – {t_end}s]")
plt.xlabel("Time (s)")
plt.ylabel("ω(t)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Unwrap phases
phi_clean = np.unwrap(x_clean)
phi_low = np.unwrap(x_low)
phi_high = np.unwrap(x_high)

# Compute phase increments
dphi_clean = np.diff(phi_clean)
dphi_low = np.diff(phi_low)
dphi_high = np.diff(phi_high)

plt.figure(figsize=(12,5))

# Overlaid histograms with counts
#plt.hist(dphi_clean, bins=100, alpha=0.6, label='Clean', color='blue')
#plt.hist(dphi_low, bins=100, alpha=0.6, label='Small noise', color='green')
plt.hist(dphi_high, bins=100, alpha=0.6, label='Large noise', color='red')

plt.xlabel("Phase increment Δφ (radians)")
plt.ylabel("Count")
plt.title("Phase increment distribution (unwrapped)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
corr_low, _ = pearsonr(x_clean, x_low)
corr_high, _ = pearsonr(x_clean, x_high)
print("Correlation with clean signal:")
print(f"Small noise: {corr_low:.3f}")
print(f"Large noise: {corr_high:.3f}")

# --- Phase unwrapping ---
phi_clean = np.unwrap(np.angle(np.exp(1j*x_clean)))
phi_low = np.unwrap(np.angle(np.exp(1j*x_low)))
phi_high = np.unwrap(np.angle(np.exp(1j*x_high)))

dphi_clean = np.diff(phi_clean)
dphi_low = np.diff(phi_low)
dphi_high = np.diff(phi_high)

# --- Plot raw signals ---
plt.figure(figsize=(14,5))
plt.plot(t[:2000], x_clean[:2000], label='Clean')
plt.plot(t[:2000], x_high[:2000], alpha=0.6, label='Noisy (large)')
plt.title("Raw signals (first 2000 points)")
plt.xlabel("Time")
plt.ylabel("Signal")
plt.legend()
plt.grid(True)
plt.show()

# --- Plot phase increment histograms ---
plt.figure(figsize=(12,5))
plt.hist(dphi_clean, bins=100, alpha=0.6, label='Clean', color='blue')
plt.hist(dphi_low, bins=100, alpha=0.6, label='Small noise', color='green')
plt.hist(dphi_high, bins=100, alpha=0.6, label='Large noise', color='red')
plt.xlabel("Phase increment Δφ")
plt.ylabel("Count")
plt.title("Phase increment distribution (unwrapped)")
plt.legend()
plt.grid(True)
plt.show()

#Tests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.integrate import solve_ivp

# -----------------------------
# 1. Precompute time grid ONCE
# -----------------------------
T = 300
fs = 5
n_points = int(T * fs)
start = int(0.05 * n_points)
end   = int(0.95 * n_points)

t = np.linspace(0, T, n_points)
dt = t[1] - t[0]


# -----------------------------
# 2. Rössler generator (faster)
# -----------------------------
def generate_rossler_x1(wx, t, a=0.165, b=0.2, c=10):
    def rossler(t, s):
        x, y, z = s
        return (-y - z, wx * x + a * y, b + z * (x - c))

    s0 = (1.0, 1.0, 1.0)

    # faster tolerances + RK45
    sol = solve_ivp(
        rossler,
        (t[0], t[-1]),
        s0,
        t_eval=t,
        rtol=1e-6,
        atol=1e-8
    )

    return sol.y[0]


# -----------------------------
# 3. Faster phase metrics
# -----------------------------
def calcular_metricas_fase(x, dt):
    analytic = hilbert(x)

    # faster than np.unwrap + diff
    phi = np.unwrap(np.angle(analytic))
    omega = np.gradient(phi) / dt

    M = omega.mean()
    S = omega.std()
    V = S / M if M != 0 else np.nan

    return V, M, S


# -----------------------------
# 4. Parameter ranges
# -----------------------------
wx_values = np.linspace(0.2, 1, 20)
noise_levels = np.linspace(0, 4, 20)

V_map = np.empty((len(noise_levels), len(wx_values)))
M_map = np.empty_like(V_map)
S_map = np.empty_like(V_map)


# -----------------------------
# 5. Study loop (less overhead)
# -----------------------------
for j, wx in enumerate(wx_values):
    #print(wx)

    x_clean = generate_rossler_x1(wx, t)


    for i, noise in enumerate(noise_levels):

        noisy_signal = x_clean + np.random.normal(0, noise, size=n_points)

        V, M, S = calcular_metricas_fase(noisy_signal[start:end], dt)

        V_map[i, j] = V
        M_map[i, j] = M
        S_map[i, j] = S


# -----------------------------
# 6. Plot heatmaps
# -----------------------------
WX, NOISE = np.meshgrid(wx_values, noise_levels)

def plot_heatmap(data, title, cmap="viridis"):
    plt.figure(figsize=(8, 5))
    pcm = plt.pcolormesh(WX, NOISE, data, shading='auto', cmap=cmap)
    plt.colorbar(pcm)
    plt.xlabel("Signal frequency (ωx)")
    plt.ylabel("Noise level (σ)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_heatmap(V_map, "Irregularity V = S/M")
plot_heatmap(M_map, "Mean phase velocity M")
plot_heatmap(S_map, "Phase velocity std S")

In [ ]:
# -----------------------------
# 7. Visualize V in M-S space
# -----------------------------
plt.figure(figsize=(8,6))

# Flatten arrays to plot in M-S space
M_flat = M_map.flatten()
S_flat = S_map.flatten()
V_flat = V_map.flatten()

# Scatter plot colored by irregularity
sc = plt.scatter(M_flat, S_flat, c=V_flat, cmap='viridis', s=60)
plt.colorbar(sc, label='Irregularity V = S/M')

plt.xlabel("Mean phase velocity M")
plt.ylabel("Phase velocity std S")
plt.title("Phase irregularity V as function of M and S")
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# 7. Sensitivity demonstration
# -----------------------------
wx_index = 10  # choose one frequency

plt.figure(figsize=(10,5))

plt.plot(noise_levels, M_map[:, wx_index], label="Mean velocity M")
plt.plot(noise_levels, S_map[:, wx_index], label="Std deviation S")
plt.plot(noise_levels, V_map[:, wx_index], label="Irregularity V=S/M")

plt.xlabel("Noise level")
plt.ylabel("Metric value")
plt.title("Sensitivity of metrics to noise")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# -----------------------------
# 7. Different Rösslers vs Noise
# -----------------------------

plt.figure(figsize=(10,6))

for j, wx_val in enumerate(wx_values):
    plt.plot(noise_levels,
             S_map[:, j],
             marker='o',
             label=f'wx = {wx_val:.2f}')

plt.xlabel("Noise level")
plt.ylabel("Standard deviation (S)")
plt.title("Effect of Noise on Different Rössler Regimes")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.signal import hilbert
import matplotlib.pyplot as plt

def rossler(t, state, a=0.2, b=0.2, c=5.7):
    x, y, z = state
    dx = -y - z
    dy = x + a*y
    dz = b + z*(x - c)
    return [dx, dy, dz]


t = np.linspace(0, 500, 50000)

sol = solve_ivp(
    rossler,
    [t[0], t[-1]],
    [1.0, 1.0, 1.0],
    t_eval=t
)

x = sol.y[0]

analytic_signal = hilbert(x)
phase = np.unwrap(np.angle(analytic_signal))

dt = t[1] - t[0]
phase_velocity = np.gradient(phase, dt)

M_clean = np.mean(phase_velocity)     # mean phase velocity
S_clean = np.std(phase_velocity)      # phase fluctuations
V_clean = S_clean / M_clean            # irregularity measure

noise_strength = 4
x_noisy = x + noise_strength * np.random.randn(len(x))

analytic_signal_noisy = hilbert(x_noisy)
phase_noisy = np.unwrap(np.angle(analytic_signal_noisy))

phase_velocity_noisy = np.gradient(phase_noisy, dt)

M_noisy = np.mean(phase_velocity_noisy)
S_noisy = np.std(phase_velocity_noisy)
V_noisy = S_noisy / M_noisy

print("NO NOISE")
print(f"M = {M_clean:.3f}, S = {S_clean:.3f}, V = {V_clean:.3f}")

print("\nWITH NOISE")
print(f"M = {M_noisy:.3f}, S = {S_noisy:.3f}, V = {V_noisy:.3f}")

plt.figure(figsize=(10, 4))
plt.plot(t[:5000], phase[:5000], label="Clean")
plt.plot(t[:5000], phase_noisy[:5000], label="Noisy", alpha=0.7)
plt.xlabel("Time")
plt.ylabel("Unwrapped phase")
plt.legend()
plt.title("Phase growth in Rössler dynamics")
plt.show()

In [ ]:
from scipy.signal import hilbert, welch
def rossler(t, state, a=0.2, b=0.2, c=5.7):
    x, y, z = state
    return [-y - z, x + a*y, b + z*(x - c)]

t = np.linspace(0, 300, 30000)
dt = t[1] - t[0]

sol = solve_ivp(
    rossler,
    [t[0], t[-1]],
    [1.0, 1.0, 1.0],
    t_eval=t
)

x_clean = sol.y[0]

noise_level = 0.1
x_noisy = x_clean + noise_level * np.random.randn(len(x_clean))

x_white = np.std(x_clean) * np.random.randn(len(x_clean))

def get_phase(x):
    return np.unwrap(np.angle(hilbert(x)))

phase_clean = get_phase(x_clean)
phase_noisy = get_phase(x_noisy)
phase_white = get_phase(x_white)

f_clean, P_clean = welch(x_clean, fs=1/dt)
f_noisy, P_noisy = welch(x_noisy, fs=1/dt)
f_white, P_white = welch(x_white, fs=1/dt)

plt.figure(figsize=(12, 4))
plt.plot(t[:3000], x_clean[:3000], label="Clean Rössler")
plt.plot(t[:3000], x_noisy[:3000], label="Noisy Rössler", alpha=0.7)
plt.plot(t[:3000], x_white[:3000], label="White noise", alpha=0.7)
plt.legend()
plt.title("Time series")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(t[:3000], phase_clean[:3000], label="Clean Rössler")
plt.plot(t[:3000], phase_noisy[:3000], label="Noisy Rössler", alpha=0.7)
plt.plot(t[:3000], phase_white[:3000], label="White noise", alpha=0.7)
plt.legend()
plt.title("Unwrapped phase")
plt.show()

plt.figure(figsize=(10, 4))
plt.semilogy(f_clean, P_clean, label="Clean Rössler")
plt.semilogy(f_noisy, P_noisy, label="Noisy Rössler")
plt.semilogy(f_white, P_white, label="White noise")
plt.xlim(0, 5)
plt.legend()
plt.title("Power spectrum")
plt.xlabel("Frequency")
plt.show()